In [ ]:
import warnings
from collections import Counter, defaultdict
from typing import DefaultDict, Iterable, Mapping, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=UserWarning)
%matplotlib inline

try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt", quiet=True)
try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab", quiet=True)

SEED = 42
RNG = np.random.default_rng(SEED)
plt.rcParams.update({"figure.figsize": (7, 3.5), "font.size": 11})


# n-граммные языковые модели

Языковая модель задаёт вероятности последовательностей токенов. В этом семинаре мы соберём **классическую n-граммную** модель по счётчикам на корпусе: оценим $P(w_t \mid \text{контекст})$ при фиксированной длине контекста, посмотрим на **перплексию**, **сглаживание Лапласа** и **Kneser–Ney**, и сгенерируем короткий текст.

Теория и обозначения подробнее — в конспекте [lm.md](lm.md). Про токенизацию и датасет **20 Newsgroups** см. [text_preprocessing_and_vectorization.ipynb](text_preprocessing_and_vectorization.ipynb) (те же четыре категории и `remove=('headers','footers','quotes')`).

In [ ]:
NEWSGROUP_CATEGORIES: tuple[str, ...] = (
    "alt.atheism",
    "soc.religion.christian",
    "comp.graphics",
    "sci.med",
)
bunch_20ng_train = fetch_20newsgroups(
    subset="train",
    categories=list(NEWSGROUP_CATEGORIES),
    shuffle=True,
    random_state=SEED,
    remove=("headers", "footers", "quotes"),
)
bunch_20ng_test = fetch_20newsgroups(
    subset="test",
    categories=list(NEWSGROUP_CATEGORIES),
    shuffle=True,
    random_state=SEED,
    remove=("headers", "footers", "quotes"),
)


Сначала разобьем сырой текст на токены (`nltk.word_tokenize`), приведем к нижнему регистру и склеим пробелами — одна строка на документ.

In [ ]:
def tokenize_document(text: str, *, lowercase: bool = True) -> str:
    raw = text.lower() if lowercase else text
    toks = word_tokenize(raw)
    return " ".join(toks)


In [ ]:
raw_train = list(bunch_20ng_train.data)
raw_test = list(bunch_20ng_test.data)
max_train, max_test = 900, 300
train_lines = [tokenize_document(t) for t in raw_train[:max_train]]
test_lines = [tokenize_document(t) for t in raw_test[:max_test]]
train_topic_ids: np.ndarray = np.asarray(
    bunch_20ng_train.target[:max_train], dtype=np.int64
)
test_topic_ids: np.ndarray = np.asarray(
    bunch_20ng_test.target[:max_test], dtype=np.int64
)
newsgroup_target_names: tuple[str, ...] = tuple(bunch_20ng_train.target_names)
y_train_full: np.ndarray = np.asarray(bunch_20ng_train.target, dtype=np.int64)
len(train_lines), len(test_lines)

In [ ]:
train_lines[7]

## N-граммы

Для начала соберем статистики по корпусу: подсчитаем, какие у нас есть n-граммы. После подсчёта n-грамм построим таблицу условных вероятностей. 

Учтем специальные токены: 
- `UNK` дополняет короткий контекст до длины $(n-1)$, 
- `EOS` добавляется в конец каждой строки (предсказываем и его). Для каждой позиции считаем префикс из $(n-1)$ токенов и обновляем счётчик «префикс → следующий токен».


baseline-оценка для вероятностей:

$$P(w_t \mid h) = \frac{C(h, w_t)}{\sum_{\hat w} C(h, \hat w)}.$$

Марковское приближение (n-граммы):

$$P(w_t \mid w_1,\ldots,w_{t-1}) \approx P(w_t \mid w_{t-n+1},\ldots,w_{t-1}).$$

Для корпуса из документов (строк токенов) оценка частоты n-граммы $(w_1,\ldots,w_n)$:

$$C(w_1,\ldots,w_n) = \sum_{\text{документы}} \sum_{\text{позиции}} \mathbf{1}[w_{i-n+1:i+1} = (w_1,\ldots,w_n)].$$


In [ ]:
UNK = "_UNK_"
EOS = "_EOS_"


def count_ngrams(
    lines: Iterable[str],
    n: int,
) -> DefaultDict[tuple[str, ...], Counter[str]]:
    counts: DefaultDict[tuple[str, ...], Counter[str]] = defaultdict(Counter)
    pad = n - 1
    for line in lines:
        toks = line.split()
        seq = list(toks)
        seq.append(EOS)
        for i in range(len(seq)):
            pref_list: list[str] = []
            for j in range(pad):
                idx = i - pad + j
                pref_list.append(UNK if idx < 0 else seq[idx])
            prefix = tuple(pref_list)
            w = seq[i]
            counts[prefix][w] += 1 # It's equivalent to the count of n-gram itself
    return counts


Посмотрим, как работает подсчет:

In [ ]:
def conditional_counts_to_ngrams(
    cond: DefaultDict[tuple[str, ...], Counter[str]],
) -> Counter[tuple[str, ...]]:
    out: Counter[tuple[str, ...]] = Counter()
    for pref, ctr in cond.items():
        for w, c in ctr.items():
            out[pref + (w,)] += c
    return out


def ngram_table(
    ctr: Counter[tuple[str, ...]],
    *,
    k: int,
) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for ng, c in ctr.most_common(k):
        rows.append({"ngram": " ".join(ng), "count": c})
    return pd.DataFrame(rows)


synth_lines = [
    "the cat sat on the mat near the door",
    "the dog ran on the beach while the sun set",
    "cats and dogs are common pets in many homes",
    "on the mat the cat slept peacefully",
]
big_synth = conditional_counts_to_ngrams(count_ngrams(synth_lines, 2))
tri_synth = conditional_counts_to_ngrams(count_ngrams(synth_lines, 3))
display(ngram_table(big_synth, k=8))
display(ngram_table(tri_synth, k=8))



То же самое проделаем на нашем датасете и заодно посмотрим, какие n-граммы типичны для разных топиков.

In [ ]:
y_name: np.ndarray = np.asarray(
    [newsgroup_target_names[i] for i in y_train_full],
    dtype=object,
)
tokenized_ng: list[str] = [tokenize_document(t) for t in bunch_20ng_train.data]
(y_train_full.shape, y_name.shape, len(tokenized_ng))

ngrams_1 = 2
ngrams_2 = 6

fig, axes = plt.subplots(
    len(newsgroup_target_names),
    2,
    figsize=(12, 14),
    constrained_layout=True,
)
for ci, cname in enumerate(newsgroup_target_names):
    short = cname.split(".")[-1]
    lines_cat = [tokenized_ng[i] for i in np.flatnonzero(y_train_full == ci)]
    c2 = count_ngrams(lines_cat, ngrams_1)
    c3 = count_ngrams(lines_cat, ngrams_2)
    big_cat = conditional_counts_to_ngrams(c2)
    tri_cat = conditional_counts_to_ngrams(c3)
    top_bi = big_cat.most_common(10)
    top_tri = tri_cat.most_common(5)
    labels_bi = [" ".join(ng) for ng, _ in top_bi]
    counts_bi = [c for _, c in top_bi]
    labels_tri = [" ".join(ng) for ng, _ in top_tri]
    counts_tri = [c for _, c in top_tri]
    axes[ci, 0].barh(range(len(labels_bi)), counts_bi, color="steelblue")
    axes[ci, 0].set_yticks(range(len(labels_bi)))
    axes[ci, 0].set_yticklabels(labels_bi, fontsize=7)
    axes[ci, 0].invert_yaxis()
    axes[ci, 0].set_title(f"{short} — {ngrams_1}-grams", fontsize=10)
    axes[ci, 0].set_xlabel("count")
    axes[ci, 1].barh(range(len(labels_tri)), counts_tri, color="coral")
    axes[ci, 1].set_yticks(range(len(labels_tri)))
    axes[ci, 1].set_yticklabels(labels_tri, fontsize=7)
    axes[ci, 1].invert_yaxis()
    axes[ci, 1].set_title(f"{short} — {ngrams_2}-grams", fontsize=10)
    axes[ci, 1].set_xlabel("count")
plt.suptitle(f"20 Newsgroups train: top {ngrams_1}-grams (left) and {ngrams_2}-grams (right) per category")
plt.show()



In [ ]:
def _normalize_prefix(prefix_tokens: Sequence[str], n: int) -> tuple[str, ...]:
    """ Паддинг до длины n-gram-1"""
    pad = n - 1
    pt = list(prefix_tokens[-pad:]) if pad else []
    if len(pt) < pad:
        pt = [UNK] * (pad - len(pt)) + pt
    return tuple(pt)

class NGramLanguageModel:
    def __init__(self, n: int) -> None:
        if n < 1:
            raise ValueError("n must be >= 1")
        self.n = n
        self.probs: DefaultDict[tuple[str, ...], dict[str, float]] = defaultdict(dict)

    def fit(self, lines: Sequence[str]):
        counts = count_ngrams(lines, self.n)
        self.probs = defaultdict(dict)
        for prefix, ctr in counts.items():
            total = sum(ctr.values())
            if total == 0:
                continue
            self.probs[prefix] = {w: ctr[w] / total for w in ctr}
        return self

    def get_possible_next_tokens(self, prefix: str) -> Mapping[str, float]:
        tokens = prefix.split()
        pref = _normalize_prefix(tokens, self.n)
        return self.probs.get(pref, {})

    def get_next_token_prob(self, prefix: str, next_token: str) -> float:
        return float(self.get_possible_next_tokens(prefix).get(next_token, 0.0))


Ниже на игрушечном корпусе оценим биграммы и выведем распределение после контекста **the** таблицей.

In [ ]:
toy_train = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat and the dog",
]
toy_train_tok = [" ".join(word_tokenize(s.lower())) for s in toy_train]
lm2 = NGramLanguageModel(n=2).fit(toy_train_tok)
dist = dict(lm2.get_possible_next_tokens("the"))
display(pd.DataFrame([{"w": k, "P(w|the)": v} for k, v in sorted(dist.items())]))

Проверим, будет ли последнее слово из n-граммы лежать в списке вероятных для n-1-граммы. 

In [ ]:
TOP_K = 10
n_ng = 4
NUM_SAMPLE_GRAMS = 4


def _top_k_probs_df(lm: NGramLanguageModel, prefix: str, k: int) -> pd.DataFrame:
    dist = dict(lm.get_possible_next_tokens(prefix))
    ranked = sorted(dist.items(), key=lambda x: x[1], reverse=True)[:k]
    return pd.DataFrame(
        [{"rank": r + 1, "w": w, "P(w|context)": p} for r, (w, p) in enumerate(ranked)]
    )


def _gram_has_word_tokens(gram: tuple[str, ...]) -> bool:
    words = {t.lower() for t in gram if any(ch.isalpha() for ch in t)}
    return len(words) >= len(gram) - 1 


def _collect_frequent_nplus1_grams(
    lines: Sequence[str],
    order_n: int,
    num_grams: int,
) -> list[tuple[str, ...]]:
    counts = count_ngrams(lines, order_n)
    big_cat = conditional_counts_to_ngrams(counts)
    top_grams = big_cat.most_common(order_n * 4)
    filtered = [gram[0]  for gram in top_grams if _gram_has_word_tokens(gram[0])]
    return filtered[::3][:order_n]


def _display_frequent_gram_windows(
    lm: NGramLanguageModel,
    model_name: str,
    order_n: int,
    *,
    top_k: int = 10,
    num_grams: int = 4,
) -> None:
    ctx = order_n - 1
    grams = _collect_frequent_nplus1_grams(train_lines, order_n, num_grams=num_grams)
    for gi, gram in enumerate(grams, start=1):
        gstr = " ".join(gram)
        display(Markdown(f"*Грамма {gi}:* `{gstr}`"))
        pref_first = " ".join(gram[:ctx]) if ctx > 0 else ""
        pref_last = " ".join(gram[-ctx:]) if ctx > 0 else ""
        print(f"окно 1 (первые n-1 токенов): `{pref_first if pref_first else '<empty>'}`")
        display(_top_k_probs_df(lm, pref_first, top_k))
        print(f"окно 2 (последние n-1 токенов): `{pref_last if pref_last else '<empty>'}`")
        display(_top_k_probs_df(lm, pref_last, top_k))
        print()


lm_ng = NGramLanguageModel(n=n_ng).fit(train_lines)
_display_frequent_gram_windows(lm_ng, "n-граммы", n_ng, top_k=TOP_K, num_grams=NUM_SAMPLE_GRAMS)


## Генерация и температура

Текст генерируется, пока не выпал `EOS` или не достигнута максимальная длина, авторегрессионно: сэмплируем $w \sim P(\cdot \mid \text{префикс})$ для каждого нового шага, постепенно добавляя в префикс нагенерированные слова. Вместо жадного argmax можно брать распределение с **температурой** $\tau > 0$ и семплировать из этого распределения:

$$\tilde p_i \propto p_i^{1/\tau}, \qquad \sum_i \tilde p_i = 1.$$

При $\tau \to 0$ чаще выбираются пики; при $\tau > 1$ распределение размывается. На синтетике сравним вероятности в таблице; на Newsgroups — короткая генерация.

In [ ]:
def get_next_token(
    lm: NGramLanguageModel,
    prefix: str,
    rng: np.random.Generator,
    *,
    temperature: float = 1.0,
) -> str:
    dist = dict(lm.get_possible_next_tokens(prefix))
    if not dist:
        return EOS
    tokens = list(dist.keys())
    probs = np.array([dist[t] for t in tokens], dtype=np.float64)
    if temperature == 0.0:
        best = int(np.argmax(probs))
        return tokens[best]
    tau = max(temperature, 1e-12)
    logits = np.log(np.clip(probs, 1e-300, 1.0)) / tau
    ex = np.exp(logits - np.max(logits))
    p = ex / ex.sum()
    idx = int(rng.choice(len(tokens), p=p))
    return tokens[idx]


На биграммной модели `lm2` и префиксе «the» сравним исходные $p_i$ и взвешенные $\tilde p_i \propto p_i^{1/\tau}$ для нескольких температур — **таблица**.

In [ ]:
prefix = "the"
temps = [0.25, 0.75, 1.5]
base = np.array(list(dist.values()), dtype=np.float64)
keys = list(dist.keys())
rows = []
for tau in temps:
    logits = np.log(np.clip(base, 1e-300, 1.0)) / max(tau, 1e-12)
    ex = np.exp(logits - np.max(logits))
    tilde = ex / ex.sum()
    for k, p, t in zip(keys, base, tilde):
        rows.append({"tau": tau, "w": k, "p": p, "tilde_p": t})
display(pd.DataFrame(rows).pivot(index="w", columns="tau", values="tilde_p").round(4))

Проверим, действительно ли слова семплируются с этими вероятностями?

In [ ]:
mc = 4000
fig, axes = plt.subplots(1, len(temps), figsize=(10, 3), sharey=True)
for ax, tau in zip(axes, temps):
    cts: dict[str, int] = {}
    for _ in range(mc):
        t = get_next_token(lm2, prefix, RNG, temperature=tau)
        cts[t] = cts.get(t, 0) + 1
    keys_mc = sorted(cts.keys())
    ax.bar(keys_mc, [cts[k] / mc for k in keys_mc], color="steelblue")
    ax.set_title(f"tau = {tau}")
    ax.tick_params(axis="x", rotation=55)
axes[0].set_ylabel("частота (MC)")
plt.suptitle(f"Сэмплирование следующего токена после '{prefix}'")
plt.tight_layout()
plt.show()

При обычной генерации многие контексты ни разу не встречались в обучении: условное распределение для префикса пустое, и `get_possible_next_tokens` возвращает `{}`. Тогда сэмплер сразу выдаёт `EOS` — цепочка обрывается. При ненулевом же распределении вероятности все равно часто очень пиковые, что приводит к повторам или обрывающимуся тексту.

Проверим, так ли это: возьмем биграммы из `train_lines`, при этом проверим два префикса — типичный (`the`) и заведомо редкий, не встречавшийся как контекст в обучении.


In [ ]:
TOP_K_PROBS = 10
N_GEN_EACH = 4
GEN_MAX_LEN = 40
TAU_SAMPLE_GEN = 0.85


def _generate_line(
    lm: NGramLanguageModel,
    prefix: str,
    rng: np.random.Generator,
    *,
    max_len: int,
    temperature: float,
) -> str:
    out = prefix
    for _ in range(max_len):
        w = get_next_token(lm, out, rng, temperature=temperature)
        out = out + " " + w
        if w == EOS:
            break
    return out.replace(EOS, "<EOS>")


def _print_model_block(
    lm: NGramLanguageModel,
    model_name: str,
    prefixes: list[str],
    tag: int,
) -> None:
    for pref in prefixes:
        print(f"{model_name}, префикс `{pref}` — топ-{TOP_K_PROBS} следующих токенов")
        display(_top_k_probs_df(lm, pref, TOP_K_PROBS))
        print(f"жадно (τ=0), {N_GEN_EACH} прогона:")
        for i in range(N_GEN_EACH):
            rr = np.random.default_rng(SEED + 100 * tag + i)
            print(f"  {i + 1}.", _generate_line(lm, pref, rr, max_len=GEN_MAX_LEN, temperature=0.0))
        print(f"семплирование (τ={TAU_SAMPLE_GEN}), {N_GEN_EACH} прогона:")
        for i in range(N_GEN_EACH):
            rr = np.random.default_rng(SEED + 100 * tag + 50 + i)
            print(
                f"  {i + 1}.",
                _generate_line(lm, pref, rr, max_len=GEN_MAX_LEN, temperature=TAU_SAMPLE_GEN),
            )
        print()


In [ ]:
n = 2
gen_baseline_ng = NGramLanguageModel(n=n).fit(train_lines)
prefixes_ng = ["the", "synchrophasotrone"]
_print_model_block(gen_baseline_ng, f"{n}-грамма", prefixes_ng, tag=1)
d_rich = gen_baseline_ng.get_possible_next_tokens(prefixes_ng[0])
d_sparse = gen_baseline_ng.get_possible_next_tokens(prefixes_ng[1])
print(f"|next после 'the'| = {len(d_rich)}, |next после редкого| = {len(d_sparse)}")
_display_frequent_gram_windows(gen_baseline_ng, "топ вероятностей", n, top_k=TOP_K_PROBS, num_grams=4)


## Перплексия

**Перплексия** измеряет, насколько модель «удивлена» данными: чем ниже, тем лучше согласование (при честном сравнении моделей). По корпусу из $N$ токенов:

$$\mathrm{PP} = \exp\left(-\frac{1}{N} \sum_{t=1}^{N} \log P(w_t \mid \ldots)\right).$$

Для вычисления используются лог-вероятности в **натуральном** логарифме, затем берется экспонента от их суммы. Для нулевых вероятностей в случае базовой модели используют нижнюю границу `min_logprob`, чтобы не получить $-\infty$.

**Важно:** это технический приём для численной устойчивости: фиктивная нижняя граница на $\log P$ не превращает нашу модель в корректную вероятностную модель на новых n-граммах. Такую перплексию нельзя напрямую сопоставлять с перплексией сглаженных моделей (Лаплас, KN), где масса на неизвестные события задана явно — для сравнения нужна одна и та же схема оценки $P(w\mid\text{контекст})$.


In [ ]:
def perplexity(
    lm: NGramLanguageModel,
    lines: Sequence[str],
    *,
    min_logprob: float = np.log(10 ** -50.0),
    log: str = "natural",
) -> float:
    total_ll = 0.0
    n_tokens = 0
    for line in lines:
        toks = line.split()
        toks.append(EOS)
        ctx: list[str] = []
        for w in toks:
            pref = " ".join(ctx)
            lp = np.log(max(lm.get_next_token_prob(pref, w), 1e-300))
            if lp < min_logprob:
                lp = min_logprob
            total_ll += lp
            n_tokens += 1
            ctx.append(w)
    if n_tokens == 0:
        return float("inf")
    mean_nll = -total_ll / n_tokens
    if log == "log2":
        mean_bits = mean_nll / np.log(2.0)
        return float(2 ** mean_bits)
    return float(np.exp(mean_nll))


Проверим, как меняется перплексия при увеличении n.

In [ ]:
orders = [1, 2, 3, 4]
pp_train = [perplexity(NGramLanguageModel(n=k).fit(toy_train_tok), toy_train_tok) for k in orders]
df_pp_toy = pd.DataFrame({"n": orders, "perplexity_train": pp_train})
display(df_pp_toy.round(4))
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(df_pp_toy["n"], df_pp_toy["perplexity_train"], "o-", color="darkgreen")
ax.set_xlabel("n (порядок n-граммы)")
ax.set_ylabel("PP (train set)")
ax.set_title("Игрушечный корпус")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Задание: Посчитайте перплексию на тестовом датасете. Сделайте вывод, меняется ли она также, как на трейне.

Фраза с редкими n-граммами: перплексия большая, но конечная при `min_logprob`.

In [ ]:
lm3_toy = NGramLanguageModel(n=5).fit(toy_train_tok)
ood = ["the jabberwock , with eyes of flame , "]
pp_ood = perplexity(lm3_toy, [tokenize_document(s) for s in ood])
print(f"perplexity (OOD фраза, токенизировано): {pp_ood:.2e}")

Проверим перплексию на тесте:

In [ ]:
pool = list(raw_train[:max_train])
tr_raw, te_raw = train_test_split(pool, test_size=0.25, random_state=SEED)
tr_lines = [tokenize_document(t) for t in tr_raw]
te_lines = [tokenize_document(t) for t in te_raw]
n = 3
baseline = NGramLanguageModel(n=n).fit(tr_lines)
pp_baseline_test = perplexity(baseline, te_lines)
print(f" {n}-gram, perplexity(test) ≈ {pp_baseline_test:.4e}")

## Сглаживание Лапласа (additive)

Чтобы не присваивать нулевую вероятность ненаблюдённым n-граммам, используют **сглаживание Лапласа** (add-$\delta$):

$$P(w \mid h) = \frac{C(h,w) + \delta}{\sum_{\hat w} C(h,\hat w) + \delta \, |V|}.$$

Cравним базовую модель и сглаженную по Лапласу:

In [ ]:
class LaplaceLanguageModel(NGramLanguageModel):
    def __init__(self, n: int, delta: float = 1.0) -> None:
        if delta <= 0:
            raise ValueError("delta must be positive")
        super().__init__(n)
        self.delta = delta

    def fit(self, lines: Sequence[str]):
        self._counts = count_ngrams(lines, self.n)
        self.vocab = {w for ctr in self._counts.values() for w in ctr}
        self.probs = defaultdict(dict)
        return self

    def get_possible_next_tokens(self, prefix: str) -> Mapping[str, float]:
        tokens = prefix.split()
        pref = _normalize_prefix(tokens, self.n)
        ctr = self._counts.get(pref)
        if not self.vocab:
            return {}
        if ctr is None or sum(ctr.values()) == 0:
            u = 1.0 / len(self.vocab)
            return {w: u for w in self.vocab}
        # recompute on the go, not store
        total = float(sum(ctr.values()) + self.delta * len(self.vocab)) 
        return {w: (ctr.get(w, 0) + self.delta) / total for w in self.vocab}

    def get_next_token_prob(self, prefix: str, next_token: str) -> float:
        tokens = prefix.split()
        pref = _normalize_prefix(tokens, self.n)
        ctr = self._counts.get(pref)
        if not self.vocab:
            return 0.0
        if ctr is None or sum(ctr.values()) == 0:
            return (1.0 / len(self.vocab)) if next_token in self.vocab else 0.0
        total = float(sum(ctr.values()) + self.delta * len(self.vocab))
        return (ctr.get(next_token, 0) + self.delta) / total


С **аддитивным сглаживанием** у каждого префикса есть ненулевое распределение по словарю, поэтому сэмплирование не останавливается из‑за пустого $P(\cdot \mid h)$ на ненаблюдённом контексте (как у базовой модели выше).


In [ ]:
gen_lap = LaplaceLanguageModel(n=2, delta=0.01).fit(train_lines)
prefixes_ng = ["the", "QWRWDAD123122"]
_print_model_block(gen_lap, "Лаплас (δ=0.01, биграмма)", prefixes_ng, tag=2)
_display_frequent_gram_windows(gen_lap, "Лаплас (δ=0.01, биграмма): частые граммы", 2, top_k=TOP_K_PROBS, num_grams=4)


Также сравним базовый метод и сглаженный на синтетике

In [ ]:
lap_toy = LaplaceLanguageModel(n=2, delta=0.5).fit(toy_train_tok)
rows = []
for w in sorted(lap_toy.vocab):
    rows.append(
        {
            "w": w,
            "baseline": dist.get(w, 0.0),
            "laplace": lap_toy.get_next_token_prob("the", w),
        }
    )
display(pd.DataFrame(rows).query("baseline > 0 or laplace > 0.01").head(12).round(6))

Задание: реализуйте схему перехода к n-1 грамме, сравните со сглаживанием Лапласа

## Kneser–Ney

Аддитивное сглаживание простое; **Kneser–Ney** вычитает абсолютный дисконт $\delta$ из ненулевых счётчиков и переносит массу на более короткий контекст (backoff). Униграмма в базе KN задаётся через продолжения: насколько часто токен появляется в новых контекстах.

Рекурсия (схематично):

$$P_{\mathrm{KN}}(w \mid h) = \frac{\max(0, C(h,w)-\delta)}{\sum_{\hat w} C(h,\hat w)} + \lambda_h \, P_{\mathrm{KN}}(w \mid h'),$$

где $h'$ — суффикс $h$ без первого токена, $\lambda_h$ подбирается из нормировки.

Ниже в коде — упрощённая учебная реализация (один порядок n-грамм, фиксированный $\delta$). Полный интерполированный Kneser–Ney с несколькими уровнями и обучаемыми дисконтами — отдельная тема, в продакшене (например, KenLM) используют именно такие схемы.

In [ ]:
class KneserNeyLanguageModel(NGramLanguageModel):
    def __init__(self, n: int, delta: float = 0.75) -> None:
        if not (0.0 < delta < 1.0):
            raise ValueError("delta should lie in (0, 1)")
        super().__init__(n)
        self.delta = delta

    def fit(self, lines: Sequence[str]):
        self._lines = list(lines)
        self._counts = [
            count_ngrams(self._lines, order) for order in range(1, self.n + 1)
        ]
        self.vocab = self._build_vocab()
        self._p_cont = self._continuation_unigram()
        self.probs = defaultdict(dict)
        return self

    def _build_vocab(self) -> set[str]:
        v: set[str] = set()
        for cts in self._counts:
            for ctr in cts.values():
                v.update(ctr.keys())
        return v

    def _continuation_unigram(self) -> dict[str, float]:
        bi = self._counts[1]
        left_of: DefaultDict[str, set[str]] = defaultdict(set)
        for (w0,), ctr in bi.items():
            for w1 in ctr:
                if ctr[w1] > 0:
                    left_of[w1].add(w0)
        numer = {w: float(len(left_of[w])) for w in self.vocab}
        tot = sum(numer.values())
        if tot <= 0:
            u = 1.0 / max(len(self.vocab), 1)
            return {w: u for w in self.vocab}
        return {w: numer.get(w, 0.0) / tot for w in self.vocab}

    def _kn_prob(self, order: int, prefix: tuple[str, ...], w: str) -> float:
        if order == 1:
            return float(self._p_cont.get(w, 1e-300))
        idx = order - 1
        cts = self._counts[idx]
        ctr = cts.get(prefix)
        if ctr is None or sum(ctr.values()) == 0:
            shorter = prefix[1:] if len(prefix) > 1 else tuple()
            return self._kn_prob(order - 1, shorter, w)
        total = float(sum(ctr.values()))
        distinct = sum(1 for cw in ctr if ctr[cw] > 0)
        discounted = max(ctr.get(w, 0) - self.delta, 0.0) / total
        lam = (self.delta * distinct) / total
        shorter = prefix[1:] if len(prefix) > 1 else tuple()
        backoff = self._kn_prob(order - 1, shorter, w)
        return discounted + lam * backoff

    def get_next_token_prob(self, prefix: str, next_token: str) -> float:
        tokens = prefix.split()
        pref = _normalize_prefix(tokens, self.n)
        return self._kn_prob(self.n, pref, next_token)

    def get_possible_next_tokens(self, prefix: str) -> Mapping[str, float]:
        tokens = prefix.split()
        pref = _normalize_prefix(tokens, self.n)
        raw = {w: self._kn_prob(self.n, pref, w) for w in self.vocab}
        s = sum(raw.values())
        if s <= 0:
            u = 1.0 / len(self.vocab)
            return {w: u for w in self.vocab}
        return {w: raw[w] / s for w in self.vocab}


In [ ]:
gen_kn_ng = KneserNeyLanguageModel(n=2, delta=0.75).fit(train_lines)
prefixes_ng = ["the", "zzzznonexistent123"]
_print_model_block(gen_kn_ng, "Kneser–Ney (δ=0.75, биграмма)", prefixes_ng, tag=3)
_display_frequent_gram_windows(gen_kn_ng, "Kneser–Ney (δ=0.75, биграмма): частые граммы", 2, top_k=TOP_K_PROBS, num_grams=4)


Сравним модели, посчитав перплексию на тесте:

In [ ]:
n = 5
mle_ng = NGramLanguageModel(n=n).fit(tr_lines)
lap_ng = LaplaceLanguageModel(n=n, delta=0.01).fit(tr_lines)
kn_ng = KneserNeyLanguageModel(n=n, delta=0.75).fit(tr_lines)
pp_mle = perplexity(mle_ng, te_lines)
pp_lap = perplexity(lap_ng, te_lines)
pp_kn = perplexity(kn_ng, te_lines)
summary = pd.DataFrame(
    [
        {"model": "baseline", "perplexity": pp_mle},
        {"model": "laplace", "perplexity": pp_lap},
        {"model": "kneser_ney", "perplexity": pp_kn},
    ]
)
display(summary.round(3))


Задание: реализуйте простую схему исправления ошибок на n-граммах. Сделайте эксперименты для ее оценки.

Задание (*): реализуйте схему Kneser–Ney с оценкой дисконтов на тренировочном датасете с помощью подcчета статистик, [например, как здесь](https://www.cs.brandeis.edu/~cs136a/CS136a_docs/goodman2001.pdf). 